# Route context calibration — 02 terrain, buffer, dwell, high-speed access

Single source of truth for every calibrated route-context value, and the
generator for `ROUTE_CONTEXT_CALIBRATION.md`. Run `01_source_extraction.ipynb`
first.

Scope is the parameters that shape **how a train runs**, not what it pays:
terrain (which feeds the energy model), the timetable buffer quota, the
minimum dwell floor, and high-speed line access. This is the only
infrastructure domain with no monetary value in it, so there is no FX table
and no price-basis escalation anywhere in this notebook.

Outputs, all generated: `data/route_context.csv`,
`data/route_context_summary.csv`, `seed/track_route_context.csv`,
`seed/track_route_context_default.csv`, `seed/sources.csv`, and
`ROUTE_CONTEXT_CALIBRATION.md`.

In [ ]:
# Route context calibration — terrain, buffer, dwell, high-speed access
#
# The CSVs under `data/` and `seed/` and the document
# ROUTE_CONTEXT_CALIBRATION.md are generated artifacts: re-run this notebook
# (after 01) to regenerate them, never hand-edit.

import csv
import math
import statistics
from dataclasses import dataclass, asdict
from pathlib import Path


def _resolve_data_dir() -> Path:
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/route_context/calib/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
SOURCES_DIR = DATA_DIR.parent / "sources"
print(f"data directory: {DATA_DIR}")

CALIBRATION_REVIEWED = "2026-08-17"

# --- provenance vocabulary -------------------------------------------------
SOURCED = "sourced"  # named document, named locator
DERIVED = "derived"  # arithmetic on other values, formula in the note
BENCHMARK = "benchmark"  # a pan-European statistic standing in for a country
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
MISSING = "missing"
NO_RAILWAY = "no_railway"

COUNTRIES = [
    "AT",
    "BE",
    "BG",
    "CH",
    "CY",
    "CZ",
    "DE",
    "DK",
    "EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IT",
    "LT",
    "LU",
    "LV",
    "MT",
    "NL",
    "NO",
    "PL",
    "PT",
    "RO",
    "SE",
    "SI",
    "SK",
    "UK",
]
NO_RAILWAY_COUNTRIES = {"CY", "MT"}

## Value record

In [ ]:
# --- value record ----------------------------------------------------------
@dataclass
class SV:
    """One calibrated value with its provenance.

    No currency and no price basis: nothing here is money. What replaces them
    is `unit`, which carries more weight than usual in this domain — m/km,
    per mille, per cent of running time and minutes per stop are four
    different things and two of them are easy to confuse (see the note on
    ascent versus ruling gradient).
    """

    country_code: str
    parameter: str
    value: float | str | None
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    note: str = ""
    low: float | None = None
    high: float | None = None

    def __post_init__(self):
        if self.status == ASSUMED and (self.low is None or self.high is None):
            raise ValueError(
                f"{self.country_code}.{self.parameter}: ASSUMED needs a band"
            )
        if self.status in (SOURCED, BENCHMARK) and not (
            self.source_id and self.locator
        ):
            raise ValueError(
                f"{self.country_code}.{self.parameter}: {self.status} needs a source and locator"
            )
        if self.status == DERIVED and not self.note:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: derived must state its formula"
            )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "note",
    "low",
    "high",
]

values: list[SV] = []

## Terrain

Cumulative ascent per kilometre, which drives energy consumption, and the
ruling gradient, which drives traction requirement. They are not
interchangeable, and only the first reaches the database.

In [ ]:
# --- Terrain ---------------------------------------------------------------
# ASCENT is cumulative ascent per km (m/km) on the MAIN-LINE NETWORK A NIGHT
# TRAIN WOULD USE — not the country's topography. Railways follow valleys, so
# mountainous countries score far below what their geography suggests.
#
# GRADIENT is the steepest sustained gradient on those corridors (per mille).
# It drives traction requirement, not energy, and reaches no database column:
# whether a heavy train needs banking assistance is a composition question
# (Composition.locos), triggered by individual ramps rather than a national
# average. It is calibrated here because it is the natural companion to ascent
# and because dropping it would lose the reasoning behind the ascent figures.
TERRAIN = {
    "NL": (0.6, 5, "Entirely flat; only the Betuweroute ramps rise"),
    "DK": (0.9, 5, "Glacial moraine, no relief of consequence"),
    "EE": (1.0, 6, "Baltic plain"),
    "LT": (1.1, 6, "Baltic plain"),
    "LV": (1.1, 6, "Baltic plain"),
    "HU": (1.3, 10, "Pannonian basin; only the Bakony and Mátra edges rise"),
    "PL": (1.4, 12, "North European Plain; Carpathian foothills only in the far south"),
    "IE": (1.6, 12, "Gentle rolling, main lines follow lowlands"),
    "FI": (1.8, 10, "Lakeland, gently undulating throughout"),
    "BE": (
        1.9,
        16,
        "Flat Flanders offset by the Ardennes foothills on the Liège–Aachen axis",
    ),
    "FR": (
        2.2,
        25,
        "Large flat basins; Massif Central, Jura and Alpine approaches are edge cases",
    ),
    "PT": (
        2.5,
        25,
        "Coastal Lisboa–Porto flat; the Beira Alta interior route is far steeper",
    ),
    "SE": (2.6, 17, "Flat in the south, rolling through Bergslagen and Norrland"),
    "DE": (
        2.8,
        25,
        "Flat north, Mittelgebirge crossings on every north–south corridor",
    ),
    "UK": (
        3.0,
        13,
        "Rolling; the WCML crosses Shap and Beattock, Highland lines steeper",
    ),
    "CZ": (
        3.2,
        20,
        "Bohemian basin ringed by hills; main lines follow the Labe and Vltava valleys",
    ),
    "RO": (
        3.8,
        20,
        "Wallachian and Moldavian plains, but the Carpathian crossing at Predeal is severe",
    ),
    "LU": (4.2, 16, "Small network sitting largely on the Oesling plateau"),
    "BG": (4.5, 25, "Balkan range bisects the country; Sofia itself sits at 550 m"),
    "ES": (
        4.8,
        20,
        "Meseta at 600–700 m — every line out of Madrid climbs to reach it",
    ),
    "IT": (
        5.0,
        30,
        "Po valley flat, but the Apennines split the peninsula on every north–south route",
    ),
    "SK": (5.2, 20, "Carpathian arc runs the length of the country"),
    "HR": (
        5.6,
        26,
        "Slavonian plain flat; the Dinaric crossings to Rijeka and Split are steep",
    ),
    "GR": (
        6.2,
        25,
        "Mountainous throughout; Athens–Thessaloniki crosses several ranges",
    ),
    "SI": (7.0, 26, "Alps and Karst in a small network — little easy ground anywhere"),
    "NO": (
        7.5,
        21,
        "Bergensbanen reaches 1,222 m; the Oslo–Göteborg axis is flat by contrast",
    ),
    "AT": (
        8.5,
        27,
        "Alps dominate — Semmering, Tauern, Arlberg — with only the Danube valley easy",
    ),
    "CH": (
        9.5,
        27,
        "Every transit crosses the Alps, though the base tunnels have cut the worst of it",
    ),
}
assert set(TERRAIN) == set(COUNTRIES) - NO_RAILWAY_COUNTRIES

# Bands on ascent. T5 is defined but no country averages into it: it applies at
# LINE level only (Beograd–Bar, the steepest Balkan and Iberian branches).
BANDS = (
    (1.5, "T1", "Flat"),
    (4.0, "T2", "Rolling"),
    (8.0, "T3", "Hilly"),
    (15.0, "T4", "Mountainous"),
    (float("inf"), "T5", "Severe"),
)

# The database column is constrained to three values, so the five model bands
# flatten: T2 and T3 both become Hilly. A documented loss of resolution, not an
# oversight — the score column carries the full detail and is what the energy
# model actually reads.
CATEGORY_DB = {
    "T1": "Flat",
    "T2": "Hilly",
    "T3": "Hilly",
    "T4": "Mountainous",
    "T5": "Mountainous",
}

SCORE_PER_M_PER_KM = 5.0
"""terrain_score = round(ascent x 5), i.e. a 0-100 scale where 20 m/km reads
100. An affine restatement of ascent, kept because the energy model's
f_terrain coefficient is calibrated against this scale."""


def _band(ascent: float) -> tuple[str, str]:
    for limit, band, name in BANDS:
        if ascent < limit:
            return band, name
    raise AssertionError("unreachable")


terrain = {}
for cc, (ascent, gradient, why) in TERRAIN.items():
    band, band_name = _band(ascent)
    terrain[cc] = {
        "ascent": ascent,
        "band": band,
        "band_name": band_name,
        "category_db": CATEGORY_DB[band],
        "score": round(ascent * SCORE_PER_M_PER_KM),
        "gradient": gradient,
        "why": why,
    }
    values.append(
        SV(
            cc,
            "terrain_ascent_m_per_km",
            ascent,
            "m/km",
            ASSUMED,
            "TN-TOPOGRAPHY",
            "corridor assessment",
            f"{why}. Band {band} ({band_name})",
            low=max(0.0, ascent - 1.0),
            high=ascent + 1.0,
        )
    )
    values.append(
        SV(
            cc,
            "ruling_gradient_permille",
            gradient,
            "per mille",
            ASSUMED,
            "TN-TOPOGRAPHY",
            "corridor assessment",
            "Steepest sustained gradient on the main corridors. Drives traction "
            "requirement, not energy, and reaches no database column",
            low=max(0.0, gradient - 5),
            high=gradient + 5,
        )
    )
    values.append(
        SV(
            cc,
            "terrain_score",
            terrain[cc]["score"],
            "score 0-100",
            DERIVED,
            "TN-TOPOGRAPHY",
            "corridor assessment",
            f"round(ascent {ascent} x {SCORE_PER_M_PER_KM:g}) = {terrain[cc]['score']}",
        )
    )
    values.append(
        SV(
            cc,
            "terrain_category",
            terrain[cc]["category_db"],
            "category",
            DERIVED,
            "TN-TOPOGRAPHY",
            "corridor assessment",
            f"band {band} flattened to the three values the DB column allows",
        )
    )

_by_band: dict[str, list[str]] = {}
for cc, t in terrain.items():
    _by_band.setdefault(t["band"], []).append(cc)
print({b: len(ccs) for b, ccs in sorted(_by_band.items())})

## Timetable buffer

The supplement that turns the router's technical running time into a
schedulable commercial one. Derived from two observable drivers per country —
and, since ROUTE_BUILDER 0.9.23 gave us `ontd.route_legs`, checkable against
what real night-train timetables actually carry.

In [ ]:
# --- Timetable buffer ------------------------------------------------------
# buffer_pct = BASE + UTIL_COEF x sqrt(u / u_EU27) + DELAY_COEF x (1 - p)
#
# The buffer is a percentage of RUNNING TIME ONLY. Dwell at commercial stops,
# locomotive changes, border and traction changes and crew changes are modelled
# separately and must not be folded in, or a route with many stops gets
# double-padded.
BASE_PCT = 4.0
UTIL_COEF = 2.5
DELAY_COEF = 6.0
UTIL_EU27 = 18.67  # k train-km per line-km, RMMS 2024 EU27 average

# RMMS covers EU27 + NO only, and has no long-distance punctuality entry for
# the Baltics or IE. Assumed inputs are flagged per country rather than hidden.
UTILISATION = {
    "AT": 30.61,
    "BE": 30.70,
    "BG": 7.85,
    "CH": 55.00,
    "CZ": 18.11,
    "DE": 29.97,
    "DK": 31.33,
    "EE": 5.81,
    "ES": 11.74,
    "FI": 8.23,
    "FR": 15.70,
    "GR": 5.31,
    "HR": 7.28,
    "HU": 14.31,
    "IE": 8.67,
    "IT": 24.78,
    "LT": 6.35,
    "LU": 27.79,
    "LV": 5.52,
    "NL": 49.37,
    "NO": 12.39,
    "PL": 13.97,
    "PT": 13.36,
    "RO": 5.23,
    "SE": 15.53,
    "SI": 18.81,
    "SK": 14.20,
    "UK": 38.00,
}
PUNCTUALITY = {
    "AT": 0.814,
    "BE": 0.878,
    "BG": 0.867,
    "CH": 0.900,
    "CZ": 0.747,
    "DE": 0.536,
    "DK": 0.851,
    "EE": 0.880,
    "ES": 0.843,
    "FI": 0.830,
    "FR": 0.842,
    "GR": 0.339,
    "HR": 0.444,
    "HU": 0.592,
    "IE": 0.900,
    "IT": 0.669,
    "LT": 0.880,
    "LU": 0.757,
    "LV": 0.880,
    "NL": 0.880,
    "NO": 0.800,
    "PL": 0.771,
    "PT": 0.534,
    "RO": 0.197,
    "SE": 0.712,
    "SI": 0.386,
    "SK": 0.746,
    "UK": 0.700,
}
ASSUMED_UTILISATION = {
    "CH": "densest network in Europe; RMMS covers EU27 + NO only",
    "UK": "RMMS covers EU27 + NO only; figure consistent with Network Rail density",
}
ASSUMED_PUNCTUALITY = {
    "CH": "best-performing network in Europe; no RMMS entry",
    "UK": "PPM-equivalent, not the RMMS definition",
    "IE": "no RMMS long-distance entry; lightly used, punctual network",
    "EE": "no RMMS long-distance entry; lightly used, punctual network",
    "LV": "no RMMS long-distance entry; lightly used, punctual network",
    "LT": "no RMMS long-distance entry; lightly used, punctual network",
    "NO": "no RMMS long-distance entry for 2022",
}

buffer_pct = {}
for cc in sorted(UTILISATION):
    u, p = UTILISATION[cc], PUNCTUALITY[cc]
    util_term = UTIL_COEF * math.sqrt(u / UTIL_EU27)
    delay_term = DELAY_COEF * (1.0 - p)
    buffer_pct[cc] = round(BASE_PCT + util_term + delay_term, 2)

    for parameter, value, unit, assumed_map, source, locator in (
        (
            "network_utilisation",
            u,
            "k train-km per line-km",
            ASSUMED_UTILISATION,
            "RMMS-9",
            "Fig.5 + Fig.69",
        ),
        ("punctuality_ld", p, "share", ASSUMED_PUNCTUALITY, "RMMS-9", "Fig.116"),
    ):
        if cc in assumed_map:
            values.append(
                SV(
                    cc,
                    parameter,
                    value,
                    unit,
                    ASSUMED,
                    source,
                    locator,
                    assumed_map[cc],
                    low=value * 0.8,
                    high=min(1.0, value * 1.2) if unit == "share" else value * 1.2,
                )
            )
        else:
            values.append(
                SV(
                    cc,
                    parameter,
                    value,
                    unit,
                    BENCHMARK,
                    source,
                    locator,
                    "RMMS 2024, data year 2022",
                )
            )
    # Kept as a reference figure and as the fallback when no ONTD snapshot is
    # loaded — NOT what gets seeded. The seeded value is the measured
    # schedule supplement below. Utilisation and punctuality are retained per
    # country for the same reason: they are exactly what a "night trains get
    # priority" scenario would move.
    values.append(
        SV(
            cc,
            "timetable_buffer_theory_pct",
            buffer_pct[cc],
            "% of running time",
            DERIVED,
            "RMMS-9",
            "Fig.5 + Fig.69 + Fig.116",
            f"{BASE_PCT} + {UTIL_COEF} x sqrt({u} / {UTIL_EU27}) + "
            f"{DELAY_COEF} x (1 - {p}) = {util_term:.2f} + {delay_term:.2f} + base",
        )
    )

_b = sorted(buffer_pct.values())
print(f"buffer {_b[0]:.1f}–{_b[-1]:.1f}%, median {statistics.median(_b):.1f}%")

# Published national practice is roughly 3-5% regular supplement plus 3-5%
# construction allowance. A calibration landing outside 4-14% would mean the
# formula, not the country, is wrong.
for cc, pct in buffer_pct.items():
    assert 4.0 <= pct <= 14.0, f"{cc}: buffer {pct}% outside published practice"

## Dwell floor and high-speed access

Two parameters that are uniform across all 28 countries today, and stay
per-country columns anyway.

In [ ]:
# --- Dwell floor and high-speed access -------------------------------------
MIN_DWELL_MIN = 2.0
HSR_ALLOWED = False

# Both are the same value for every country. They stay PER-COUNTRY columns
# rather than collapsing into the defaults row for one reason: a scenario
# overrides per country, so keeping the column is what lets a scenario ask
# "what if Spanish high-speed access opened" or "what if German dwell is really
# four minutes" without a schema change. The uniform value is the calibration;
# the column is the lever.
for cc in sorted(TERRAIN):
    values.append(
        SV(
            cc,
            "min_dwell_min",
            MIN_DWELL_MIN,
            "min/stop",
            ASSUMED,
            "TN-TOPOGRAPHY",
            "sleeper dispatch practice",
            "Floor, not a value: where a stop exists for another reason — "
            "locomotive or traction change, crew change, reversal, border "
            "control, watering — the longer operational requirement REPLACES "
            "it rather than adding to it. Additive, never multiplied by the "
            "buffer: dwell is not running time",
            low=1.0,
            high=4.0,
        )
    )
    values.append(
        SV(
            cc,
            "hsr_allowed",
            HSR_ALLOWED,
            "flag",
            ASSUMED,
            "TN-TOPOGRAPHY",
            "target network scope",
            "No country's high-speed lines are assumed open to a loco-hauled "
            "night train in the base scenario. ANDed with the composition's own "
            "hsr_allowed, so a high-speed-capable composition still needs the "
            "country to permit it",
            low=0.0,
            high=1.0,
        )
    )
print(
    f"dwell floor {MIN_DWELL_MIN} min, hsr_allowed {HSR_ALLOWED}, "
    f"uniform across {len(TERRAIN)} countries"
)

## The ONTD check

What real night-train timetables imply, read against what the formula
predicts. Written by `01`; absent until that notebook has run against a
database with a loaded ONTD snapshot.

In [ ]:
# --- ONTD observation set ---------------------------------------------------
# Read, never written here: 01 extracts it against a live database, this cell
# only consumes what is on disk. So the calibration stays reproducible on a
# machine with no ONTD snapshot — the document then says the check is pending
# instead of silently omitting it.
ONTD_BY_COUNTRY_CSV = SOURCES_DIR / "ontd_buffer_by_country.csv"

ontd_observed: dict[str, dict] = {}
if ONTD_BY_COUNTRY_CSV.is_file():
    with open(ONTD_BY_COUNTRY_CSV, newline="", encoding="utf-8-sig") as fh:
        for row in csv.DictReader(fh):
            ontd_observed[row["country_code"]] = row
    print(f"observation set: {len(ontd_observed)} countries")
else:
    print(
        "no observation set at sources/ontd_buffer_by_country.csv — run 01's "
        "last cell against a database with a loaded ONTD snapshot. The buffer "
        "section of the document will report the check as pending."
    )


def _observed(cc: str, key: str) -> float | None:
    row = ontd_observed.get(cc)
    if not row or not row.get(key):
        return None
    return float(row[key])


# --- The schedule supplement ------------------------------------------------
# ONE multiplier per country on the router's pure passage time, containing
# everything that makes a real timetable slower than the router:
#
#   - construction and pathing allowances the infrastructure manager applies
#   - margin for a night train not always having priority
#   - speed the train cannot hold — curves, junctions, temporary restrictions
#   - acceleration and braking the dynamics model misses (it fires once after
#     a stop and once before the next, not at every speed change in between)
#
# These were separated in an earlier draft into a "buffer quota" and a "speed
# realisation factor". That split was ABANDONED deliberately: per country we
# have one measurement and there is no second observable that separates the
# planners' margin from the router's optimism, so any split was a modelling
# choice dressed as a calibration. One value, honestly named, until better
# data exists.
#
#     scheduled_running_time = pure_passage_time x (1 + supplement)
#
# Dwell stays outside it, as it always has: a stop is not running time.
SUPPLEMENT_SHRINKAGE_LEGS = 10
"""Shrinkage strength, in legs. A country's measurement is trusted in
proportion to the evidence behind it: at 110 legs Germany is 92% itself, at 3
legs Norway is 23% itself and the rest European mean. A hard sample-size
cut-off would instead snap a 7-leg country from its own value to the mean on
one extra leg, which is a worse answer than a smooth one."""

_observed_pairs = [
    (_observed(cc, "implied_quota_pct"), _observed(cc, "n_legs") or 0)
    for cc in sorted(buffer_pct)
    if _observed(cc, "implied_quota_pct") is not None
]
SUPPLEMENT_PRIOR = (
    round(
        sum(v * n for v, n in _observed_pairs) / sum(n for _, n in _observed_pairs), 1
    )
    if _observed_pairs
    else None
)
"""Leg-weighted mean across every usable leg — the European figure, and the
fallback for a country with no night trains in the ONTD snapshot."""

supplement_pct: dict[str, float] = {}
for cc in sorted(buffer_pct):
    measured, n = _observed(cc, "implied_quota_pct"), _observed(cc, "n_legs") or 0
    if SUPPLEMENT_PRIOR is None:
        # No observation set at all: fall back to the theoretical formula, so
        # the calibration still produces a usable value on a machine with no
        # ONTD snapshot. The document says which of the two is in force.
        supplement_pct[cc] = buffer_pct[cc]
        values.append(
            SV(
                cc,
                "schedule_supplement_pct",
                buffer_pct[cc],
                "% of passage time",
                DERIVED,
                "RMMS-9",
                "Fig.5 + Fig.69 + Fig.116",
                "No ONTD observation set — theoretical buffer formula standing in. "
                "Re-run 01 against a loaded snapshot for the measured value",
            )
        )
        continue
    if measured is None:
        supplement_pct[cc] = SUPPLEMENT_PRIOR
        values.append(
            SV(
                cc,
                "schedule_supplement_pct",
                SUPPLEMENT_PRIOR,
                "% of passage time",
                ASSUMED,
                "ONTD-SNAPSHOT",
                "route_legs residual",
                "No night train in the ONTD snapshot runs through this country — "
                "European leg-weighted mean",
                low=round(SUPPLEMENT_PRIOR * 0.7, 1),
                high=round(SUPPLEMENT_PRIOR * 1.4, 1),
            )
        )
        continue
    weight = n / (n + SUPPLEMENT_SHRINKAGE_LEGS)
    supplement_pct[cc] = round(weight * measured + (1 - weight) * SUPPLEMENT_PRIOR, 1)
    values.append(
        SV(
            cc,
            "schedule_supplement_pct",
            supplement_pct[cc],
            "% of passage time",
            DERIVED,
            "ONTD-SNAPSHOT",
            "route_legs residual",
            f"{measured:.1f}% measured over {int(n)} legs, shrunk toward the "
            f"European mean {SUPPLEMENT_PRIOR}% at weight {weight:.2f} "
            f"(n / (n + {SUPPLEMENT_SHRINKAGE_LEGS}))",
        )
    )

_s = sorted(supplement_pct.values())
print(
    f"schedule supplement {_s[0]:.1f}–{_s[-1]:.1f}%, European mean {SUPPLEMENT_PRIOR}%"
)

# A measured supplement outside 15-120% means the extraction or the router
# changed shape, not that a country did. Only checked in measured mode: the
# no-snapshot fallback is the theoretical buffer, which lives at 6-10% by
# construction and would fail a bound written for the other quantity.
if SUPPLEMENT_PRIOR is not None:
    for cc, pct in supplement_pct.items():
        assert 15.0 <= pct <= 120.0, f"{cc}: supplement {pct}% implausible"

# Does the empirical residual track the two drivers the formula is built on?# Does the empirical residual track the two drivers the formula is built on?
# If it does, the residual is mostly buffer and the formula can be recalibrated
# against it. If it does not, we have mostly measured router speed error — and
# that is the finding, not a nuisance to average away.
_pairs = [
    (
        UTILISATION[cc],
        PUNCTUALITY[cc],
        _observed(cc, "implied_quota_pct"),
        _observed(cc, "n_legs"),
    )
    for cc in sorted(UTILISATION)
    if _observed(cc, "implied_quota_pct") is not None
]
_solid = [(u, p, q) for u, p, q, n in _pairs if (n or 0) >= 3]
ONTD_CORRELATIONS = {}
if len(_solid) > 2:
    us, ps, qs = zip(*_solid)
    ONTD_CORRELATIONS = {
        "n_countries": len(_solid),
        "r_utilisation": round(statistics.correlation(us, qs), 2),
        "r_punctuality": round(statistics.correlation(ps, qs), 2),
        "median_observed": round(statistics.median(qs), 1),
        "median_modelled": round(
            statistics.median([buffer_pct[cc] for cc in UTILISATION]), 1
        ),
        "median_applied": round(
            statistics.median(
                [
                    v
                    for v in (
                        _observed(cc, "applied_buffer_pct")
                        for cc in sorted(UTILISATION)
                    )
                    if v is not None
                ]
                or [0.0]
            ),
            1,
        ),
    }
    print(ONTD_CORRELATIONS)

## Export

In [ ]:
# --- Export ----------------------------------------------------------------


def write_csv(path: Path, columns: list[str], rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {path.name}: {len(rows)} rows")


DB_PARAMETERS = [
    "schedule_supplement_pct",
    "terrain_score",
    "terrain_category",
    "timetable_buffer_pct",
    "min_dwell_min",
    "hsr_allowed",
]

full_grid = list(values)
for cc in NO_RAILWAY_COUNTRIES:
    for parameter in DB_PARAMETERS:
        full_grid.append(
            SV(cc, parameter, None, "", NO_RAILWAY, note="No railway network")
        )

write_csv(DATA_DIR / "route_context.csv", SV_FIELDS, [asdict(v) for v in full_grid])

SUMMARY_COLUMNS = [
    "country_code",
    "terrain_ascent_m_per_km",
    "terrain_band",
    "terrain_category_db",
    "terrain_score",
    "ruling_gradient_permille",
    "network_utilisation",
    "punctuality_ld",
    "timetable_buffer_pct",
    "ontd_implied_buffer_pct",
    "ontd_n_legs",
    "min_dwell_min",
    "hsr_allowed",
    "inputs_assumed",
]
summary = []
for cc in sorted(TERRAIN):
    t = terrain[cc]
    assumed = [
        name
        for name, m in (
            ("utilisation", ASSUMED_UTILISATION),
            ("punctuality", ASSUMED_PUNCTUALITY),
        )
        if cc in m
    ]
    summary.append(
        {
            "country_code": cc,
            "terrain_ascent_m_per_km": t["ascent"],
            "terrain_band": t["band"],
            "terrain_category_db": t["category_db"],
            "terrain_score": t["score"],
            "ruling_gradient_permille": t["gradient"],
            "network_utilisation": UTILISATION[cc],
            "punctuality_ld": PUNCTUALITY[cc],
            "timetable_buffer_theory_pct": buffer_pct[cc],
            "ontd_implied_buffer_pct": _observed(cc, "implied_quota_pct"),
            "ontd_n_legs": _observed(cc, "n_legs"),
            "ontd_applied_buffer_pct": _observed(cc, "applied_buffer_pct"),
            "schedule_supplement_pct": supplement_pct[cc],
            "min_dwell_min": MIN_DWELL_MIN,
            "hsr_allowed": HSR_ALLOWED,
            "inputs_assumed": "+".join(assumed),
        }
    )
write_csv(DATA_DIR / "route_context_summary.csv", SUMMARY_COLUMNS, summary)

by_status: dict[str, int] = {}
for v in full_grid:
    by_status[v.status] = by_status.get(v.status, 0) + 1
print()
for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
    print(f"    {s:11} {n:4}")

_register_path = DATA_DIR / "sources_register.csv"
with open(_register_path, encoding="utf-8") as fh:
    register = {r["source_id"]: r for r in csv.DictReader(fh)}
cited = {v.source_id for v in full_grid if v.source_id}
assert cited <= set(register), (
    f"cited but not in the register: {sorted(cited - set(register))}"
)
print(f"\n  provenance: {len(cited)} sources cited, all present in the register")

## Seed export

One row per country. Buffer travels as a fraction rather than a percentage —
`track_buffer_quota_per` is what the router multiplies by, and doing the
division here means no consumer has to remember which form it holds.

In [ ]:
# --- Seed export -----------------------------------------------------------
SEED_DIR = DATA_DIR.parent / "seed"
SEED_DIR.mkdir(exist_ok=True)

TRACK_ROUTE_CONTEXT_COLUMNS = [
    "country_code",
    "track_terrain_category",
    "track_terrain_score",
    "track_buffer_quota_per",
    "track_min_boarding_time",
    "track_min_alighting_time",
    "track_hsr_allowed",
    "source_id",
    "change_log",
]

# Boarding and alighting are separate columns holding one calibrated floor.
# Both get it: the timetable takes the max of the two against the
# composition's own figures, so writing the floor once to each keeps that
# comparison honest rather than making one side dominate by construction.
route_context_seed = [
    {
        "country_code": cc,
        "track_terrain_category": terrain[cc]["category_db"],
        "track_terrain_score": terrain[cc]["score"],
        # The existing column, carrying the all-in schedule supplement. No
        # schema change: track_buffer_quota_per is already the multiplier the
        # router applies to passage time, and the legacy placeholders it held
        # (0.30-0.50) were doing this job uncalibrated.
        "track_buffer_quota_per": round(supplement_pct[cc] / 100.0, 6),
        "track_min_boarding_time": MIN_DWELL_MIN,
        "track_min_alighting_time": MIN_DWELL_MIN,
        "track_hsr_allowed": HSR_ALLOWED,
        "source_id": "RMMS-9",
        "change_log": (
            "Terrain from TN-TOPOGRAPHY corridor assessment; schedule "
            f"supplement {supplement_pct[cc]:.1f}% "
            + (
                f"from {int(_observed(cc, 'n_legs'))} ONTD legs "
                f"({_observed(cc, 'implied_quota_pct'):.1f}% measured)."
                if _observed(cc, "implied_quota_pct") is not None
                else "from the European ONTD mean — no legs for this country."
            )
        ),
    }
    for cc in sorted(TERRAIN)
]

# The fallback row: a country the model routes through but the calibration has
# no figures for. Terrain takes the MEDIAN band rather than the mean ascent —
# a mean over a range that runs from Dutch polder to Swiss Alps describes
# nowhere. Buffer takes the median quota, which is also close to what the
# formula returns at EU27-average utilisation and punctuality.
_median_score = round(statistics.median([t["score"] for t in terrain.values()]))
default_route_context = {
    "country_code": "_default",
    "track_terrain_category": "Hilly",
    "track_terrain_score": _median_score,
    "track_buffer_quota_per": round(
        (
            SUPPLEMENT_PRIOR
            if SUPPLEMENT_PRIOR is not None
            else statistics.median(list(buffer_pct.values()))
        )
        / 100.0,
        6,
    ),
    "track_min_boarding_time": MIN_DWELL_MIN,
    "track_min_alighting_time": MIN_DWELL_MIN,
    "track_hsr_allowed": HSR_ALLOWED,
}
print(
    f"  default: score {_median_score}, buffer "
    f"{default_route_context['track_buffer_quota_per']:.4f}"
)

write_csv(
    SEED_DIR / "track_route_context_default.csv",
    TRACK_ROUTE_CONTEXT_COLUMNS,
    [default_route_context],
)
write_csv(
    SEED_DIR / "track_route_context.csv",
    TRACK_ROUTE_CONTEXT_COLUMNS,
    route_context_seed,
)

SOURCE_SEED_COLUMNS = ["source_id", "source_description", "source_url", "source_date"]
write_csv(
    SEED_DIR / "sources.csv",
    SOURCE_SEED_COLUMNS,
    [
        {
            "source_id": sid,
            "source_description": (
                f"{register[sid]['title']} — {register[sid]['publisher']} "
                f"({register[sid]['pub_year']})"
            ),
            "source_url": register[sid]["url_or_file"],
            "source_date": register[sid]["date_accessed"],
        }
        for sid in sorted(cited)
    ],
)

## Document generation

In [ ]:
# --- ROUTE_CONTEXT_CALIBRATION.md generation --------------------------------
from datetime import date

DOC_PATH = DATA_DIR.parent / "ROUTE_CONTEXT_CALIBRATION.md"

# Energy factors for the band table: a 600 t train on a ~14 kWh/train-km flat
# baseline at 1.43 Wh per tonne per metre of ascent, net of regenerative
# recovery at night. Shown so the bands have a cost meaning; the consumption
# model computes its own.
REF_TRAIN_T, REF_FLAT_KWH_KM, WH_PER_T_PER_M = 600.0, 14.0, 1.43


def _energy_factor(ascent: float) -> float:
    return 1.0 + (ascent * REF_TRAIN_T * WH_PER_T_PER_M / 1000.0) / REF_FLAT_KWH_KM


BAND_ROWS = "\n".join(
    f"| **{band}** | {name} | {lo} – {hi} | {_energy_factor(lo):.2f}–{_energy_factor(hi):.2f} |"
    for (lo, hi), (band, name) in zip(
        ((0.0, 1.5), (1.5, 4.0), (4.0, 8.0), (8.0, 15.0)),
        (("T1", "Flat"), ("T2", "Rolling"), ("T3", "Hilly"), ("T4", "Mountainous")),
    )
)

TERRAIN_ROWS = "\n".join(
    f"| {cc} | {terrain[cc]['ascent']} | **{terrain[cc]['band']}** | "
    f"{terrain[cc]['category_db']} | {terrain[cc]['score']} | "
    f"{terrain[cc]['gradient']} | {terrain[cc]['why']} |"
    for cc in sorted(TERRAIN, key=lambda c: terrain[c]["ascent"])
)

BUFFER_ROWS = "\n".join(
    f"| {cc} | {UTILISATION[cc]} | {PUNCTUALITY[cc]:.3f} | "
    f"{UTIL_COEF * math.sqrt(UTILISATION[cc] / UTIL_EU27):.2f} | "
    f"{DELAY_COEF * (1 - PUNCTUALITY[cc]):.2f} | **{buffer_pct[cc]:.1f}** | "
    f"{('+'.join(n for n, m in (('utilisation', ASSUMED_UTILISATION), ('punctuality', ASSUMED_PUNCTUALITY)) if cc in m)) or 'RMMS'} | "
    f"{('%.1f' % _observed(cc, 'implied_quota_pct')) if _observed(cc, 'implied_quota_pct') is not None else '—'} | "
    f"{('%d' % _observed(cc, 'n_legs')) if _observed(cc, 'n_legs') is not None else '—'} | "
    f"{('%.1f' % _observed(cc, 'applied_buffer_pct')) if _observed(cc, 'applied_buffer_pct') is not None else '—'} |"
    for cc in sorted(buffer_pct, key=lambda c: buffer_pct[c])
)

SPEED_ROWS = "\n".join(
    f"| {cc} | {int(_observed(cc, 'n_legs') or 0) or '—'} | "
    f"{('%.1f' % _observed(cc, 'implied_quota_pct')) if _observed(cc, 'implied_quota_pct') is not None else '—'} | "
    f"{(_observed(cc, 'n_legs') or 0) / ((_observed(cc, 'n_legs') or 0) + SUPPLEMENT_SHRINKAGE_LEGS):.2f} | "
    f"**{supplement_pct[cc]:.1f} %** | **{supplement_pct[cc] / 100:.3f}** |"
    for cc in sorted(supplement_pct, key=lambda c: supplement_pct[c])
)

_METHOD_KINDS = {"standard", "official_report", "model_output"}


def _source_row(sid: str) -> str:
    r = register[sid]
    link = r["url_or_file"]
    shown = f"[link]({link})" if link.startswith("http") else f"`{link}`"
    return (
        f"| `{sid}` | {r['title']} | {r['publisher']} | {r['pub_year']} | "
        f"{r['data_year'] or '—'} | {shown} |"
    )


SOURCE_ROWS = "\n".join(
    _source_row(s) + (" " if s in cited else "")  # keep the row shape identical
    for s in sorted(register)
)
"""Every register row, not only the cited ones: three of the five underpin the
METHOD rather than a per-country value (the UIC framework, the ONTD snapshot
the check reads, the router that produced the comparison times), and a source
table that hid them would make the calibration look better sourced than it
is. seed/sources.csv keeps the cited two — that is an FK list, not a
bibliography."""

_status_counts = {
    s: sum(1 for v in full_grid if v.status == s)
    for s in (SOURCED, DERIVED, BENCHMARK, ASSUMED, MISSING, NO_RAILWAY)
}

if ONTD_CORRELATIONS:
    _verdict = (
        "the residual tracks the drivers, so it is mostly buffer and the "
        "formula's coefficients can be refitted against it"
        if (
            ONTD_CORRELATIONS["r_utilisation"] > 0.3
            and ONTD_CORRELATIONS["r_punctuality"] < -0.3
        )
        else "the residual does NOT track either driver, so it is mostly "
        "router speed error — the right response is to fix the passage-time "
        "model, not to move the buffer quotas"
    )
    ONTD_SECTION = (
        "The observation set is in place: **@@ONTD_N@@ countries** with at "
        "least three usable legs.\n\n"
        "| | value |\n|---|---|\n"
        f"| median implied supplement, ONTD legs | **{ONTD_CORRELATIONS['median_observed']}%** |\n"
        f"| median modelled buffer, this calibration | {ONTD_CORRELATIONS['median_modelled']}% |\n"
        f"| median buffer the router already applies | {ONTD_CORRELATIONS['median_applied']}% |\n"
        f"| correlation implied vs utilisation | r = {ONTD_CORRELATIONS['r_utilisation']:+.2f} (expected positive) |\n"
        f"| correlation implied vs punctuality | r = {ONTD_CORRELATIONS['r_punctuality']:+.2f} (expected negative) |\n\n"
        "**Read the two correlations before the levels.** On this run "
        f"{_verdict}.\n\n"
        "The implied supplement is measured against the router's PURE passage "
        "time — driving plus traction dynamics, with the router's own padding "
        "deliberately excluded, since that padding is the very thing being "
        "calibrated. It therefore contains buffer AND any error in the "
        "passage-time model, and the two cannot be separated by this "
        "measurement alone; the correlations are what discriminate them.\n\n"
        "The third row is a diagnostic, not an input: it is what the router "
        "currently adds to these same legs, and it is nowhere near the "
        "modelled quota in the second row. That is **not** a router bug — the "
        "router applies `track_buffer_quota_per` correctly, and the database "
        "still carries the legacy hardcoded placeholders (0.30 to 0.50 in "
        "`db/dev/seed.py`) because this calibration has never been seeded "
        "(ROADMAP §4.3).\n\n"
        "**Which is the trap to avoid when it is.** Those placeholders were "
        "implicitly compensating for the router's speed optimism, and they "
        "were close: they deliver 1.399x pure passage time where reality "
        "needs 1.508x, so trips today run about 8% fast. Seeding the "
        "calibrated 6-10% buffer *on its own* would deliver 1.081x — leaving "
        "every trip in the tool roughly 40% too fast, with a semantically "
        "correct buffer column bought at the cost of every duration. The "
        "speed realisation factor below is what carries the rest.\n\n"
        "The per-leg detail is in `sources/ontd_buffer_legs.csv`, with a named "
        "reason for every rejected leg. Legs shorter than "
        "@@MIN_LEG_MIN@@ minutes are excluded — on a six-minute leg two "
        "minutes of platform dispatch reads as a 33% supplement while saying "
        "nothing about running-time margin."
    )
else:
    ONTD_SECTION = (
        "**Not yet extracted.** The check needs a database with a loaded ONTD "
        "snapshot and a rebuilt `ontd.route_legs`. Run the last cell of "
        "`01_source_extraction.ipynb`, then re-run this notebook: the master "
        "table gains an implied-buffer column per country and this section "
        "gains the two correlations. Until then every buffer quota in this "
        "document is the formula's output with no empirical check behind it."
    )
print("document inputs assembled:", {k: v for k, v in _status_counts.items() if v})

In [ ]:
BODY = r"""
## What belongs here

The parameters that shape how a train **runs**, as opposed to what it pays:

| Parameter | DB column | Consumed by | Effect |
|---|---|---|---|
| Terrain score | `track_terrain_score` | `models/energy/calc_energy_consumption.py` | the `f_terrain × score` term in kWh/train-km |
| Terrain category | `track_terrain_category` | display only | the three-value flattening of the five bands |
| Timetable buffer | `track_buffer_quota_per` | `models/route/routing/rail_router.py`, `dynamics.py` | padding minutes per country leg — and so trip duration, and so the layover the facility charge is priced from |
| Dwell floor | `track_min_boarding_time`, `_alighting_time` | `models/route/timetable.py` | minimum scheduled dwell per commercial stop |
| High-speed access | `track_hsr_allowed` | route builder, ANDed with the composition's own flag | whether high-speed lines may be used |

Not here: the ruling gradient, which is calibrated below but reaches no
column — whether a heavy train needs banking assistance is a composition
question, triggered by individual ramps rather than a national average.

**This is the only infrastructure domain with no money in it.** No currency
conversion, no price basis, no escalation to the evaluation year. What replaces
that discipline is unit discipline: m/km, per mille, per cent of running time
and minutes per stop are four different things, and the first two are the
easiest pair in this repository to confuse.

---

## 1. Terrain

Two independent quantities, and conflating them is the classic error:

| | |
|---|---|
| **Ascent `A`** | Cumulative ascent per kilometre, m/km — every positive elevation change along a route divided by its length. **This is what drives energy consumption.** |
| **Ruling gradient `G`** | Steepest sustained gradient on the country's main corridors, ‰. **Drives traction requirement, not energy.** |

A rolling country climbs constantly but never steeply (high `A`, low `G`); a
flat country with one mountain crossing has the reverse. Both are calibrated;
only `A` reaches the database.

| Band | Name | `A` (m/km) | Energy factor, @@REF_TRAIN_T@@ t train |
|---|---|---|---|
@@BAND_ROWS@@

**No country averages T5** (> 15 m/km). T5 exists at line level only —
Beograd–Bar, the steepest Balkan and Iberian branches, rack-assisted sections.
If the target network ever extends past the EU/CH/UK perimeter, Montenegro and
Bosnia enter that band.

`terrain_score` is `round(A × @@SCORE_PER_M@@)`, a 0–100 restatement of ascent
on which 20 m/km reads 100. It exists because the energy model's `f_terrain`
coefficient is calibrated against that scale, not because it adds information.

### Per-country calibration

Scores describe the **main-line network a night train would realistically
use**, not the country's topography. Railways follow valleys, so mountainous
countries score far below what their geography suggests — and the Danube
corridor through Austria is easier than the Ardennes crossing in Belgium.

| Country | `A` (m/km) | Band | DB category | Score | `G` (‰) | What sets the level |
|---|---|---|---|---|---|---|
@@TERRAIN_ROWS@@

### Basis, and why these are not measurements

These are **judgement-based estimates from network topography**, assigned by
working through each country's main corridors against known summit elevations,
valley routings and published ruling gradients. They are calibrated to be
right in **ranking and band**, good to roughly ±1 m/km within a band. They are
not measurements and must not be quoted as such — every one carries status
`assumed` in `data/route_context.csv` for exactly that reason.

**Where the country average misleads.** For six countries the national figure
hides a spread wide enough to change the assessment:

- **Portugal** — coastal Lisboa–Porto is T1; the Beira Alta line to the Spanish border is T3/T4.
- **Norway** — Oslo–Göteborg is T1; Oslo–Bergen is solidly T4. The national 7.5 describes neither.
- **Croatia** — Slavonian plain T1, Zagreb–Rijeka T4.
- **Spain** — the high-speed network is engineered flat with long tunnels while the conventional network climbs onto the Meseta. Gauge forces cross-border night trains onto the HS network, so the *effective* Spanish score is below 4.8.
- **Switzerland** — a Basel–Chiasso transit through the Gotthard base tunnel is roughly T2/T3, far below the national 9.5. Routing choice, not geography.
- **Austria** — the Danube corridor Wien–Linz–Salzburg is T2; anything crossing the Alps southward is T4.

**Why the country average stays anyway.** The obvious fix is terrain per
segment from the routed geometry — and the router does not carry elevation, so
there is no better source in the tool. The alternative is not "finer terrain",
it is *no terrain term at all*. A national ascent average is a proxy for how
mountainous a country's network is, feeding one term of a consumption model
that is itself still a flat 28 kWh/train-km placeholder. Calibrating terrain
finer than the model it feeds would be false precision. Two things would
replace it: an elevation-enabled routing profile with tunnel and bridge
interpolation, or a one-off DEM pass over the corridors in the ONTD database —
the second is cheaper and validates this table directly against routes that
exist.

---

## 2. Timetable buffer

The routing engine returns a **technical minimum running time** from
infrastructure `maxspeed` and the composition's traction and braking. No real
timetable is built on that figure; European practice adds running-time
supplements so a schedule absorbs ordinary disturbance without cascading:

```
scheduled_running_time = technical_running_time × (1 + buffer_pct)
scheduled_total_time   = scheduled_running_time + Σ dwell + Σ operational_stops
```

**The buffer is a percentage of running time only.** Dwell, locomotive
changes, border and traction changes and crew changes are modelled separately
and must not be folded in — otherwise a route with many stops is
double-padded. In UIC 451-1 terms the quota covers the *regular running
supplement* plus the *pathing and construction allowance*. It does not cover
recovery from large disruption, which belongs to a resilience scenario rather
than the base timetable.

### The formula

```
buffer_pct = @@BASE_PCT@@ + @@UTIL_COEF@@ × √(utilisation / @@UTIL_EU27@@) + @@DELAY_COEF@@ × (1 − punctuality)
```

| Term | Source | Reasoning |
|---|---|---|
| Base @@BASE_PCT@@ pp | UIC 451-1 practice | Irreducible regular supplement; even an empty, perfectly punctual network needs margin for driving-style variance and speed restrictions |
| Utilisation, √ | RMMS Fig.5 + Fig.69 | Conflict probability rises with traffic density but **sub-linearly** — doubling density does not double the pathing margin needed, hence the square root |
| Delay, linear | RMMS Fig.116 | Low observed punctuality means realised disturbance exceeds what the incumbent's supplements absorb; a new service planned to a target reliability needs more margin there |

**On causality.** Punctuality is an outcome, not a cause, and is used
deliberately as a *proxy for realised disturbance*. A country can score badly
because it under-pads (Germany) or because its infrastructure is in poor
condition (Romania); both raise the padding a new operator needs to hold a
published arrival time, so the direction of the adjustment is the same. It
changes the confidence, not the sign.

### The theoretical formula, kept as a reference

This is what the two RMMS drivers predict a pure timetable supplement to be.
**It is not what gets seeded** — §3 explains why the measured schedule
supplement replaced it — but it is retained per country, because it is the
term a priority-improvement scenario would act on.

| Country | Utilisation | Punctuality LD | + util | + delay | Theory buffer % | Basis | ONTD implied % | legs | router applies % |
|---|---|---|---|---|---|---|---|---|---|
@@BUFFER_ROWS@@

Median **@@BUFFER_MEDIAN@@ %**, range @@BUFFER_MIN@@–@@BUFFER_MAX@@ %. That
band sits inside published national practice — roughly 3–5 % regular
supplement plus 3–5 % construction allowance — which is the first indication
the formula is not producing nonsense.

**Two results worth noticing.** The drivers push in opposite directions often
enough that the ranking is not "rich west low, poor east high": the
**Netherlands and Switzerland need high buffers despite excellent
punctuality**, purely because their networks are the densest in Europe, while
**Bulgaria and the Baltics come out lowest** because an empty network generates
few conflicts. **Germany at 10 % is the outlier among high-income countries** —
Dutch-level density combined with the worst long-distance punctuality in
western Europe (53.6 % in 2022), and both terms add.

**Assumed inputs.** RMMS covers EU27 + NO only, so CH and UK have no
utilisation or punctuality entry, and IE, EE, LV, LT and NO have no
long-distance punctuality entry. Each is flagged in the Basis column and
carries status `assumed` with a band in `data/route_context.csv`.

---

## 3. What the real timetables say

@@ONTD_SECTION@@

### One value per country, not two

An earlier draft split this residual into a buffer quota and a speed
realisation factor. **That split was abandoned, deliberately.** Per country we
have one measurement and two unknowns, and no second observable separates the
planners' margin from the router's optimism — so any split was a modelling
choice presented as a calibration.

What is seeded instead is **one schedule supplement per country**, containing
everything that makes a real timetable slower than the router's passage time:

- construction and pathing allowances the infrastructure manager applies;
- margin because a night train does not always hold priority;
- speed the train cannot sustain — curves, junctions, temporary restrictions;
- acceleration and braking the dynamics model misses, since it fires once
  after a stop and once before the next, not at every speed change between.

```
scheduled_running_time = pure_passage_time × (1 + supplement)
```

Dwell stays outside it, as always: a stop is not running time.

**Each country is trusted in proportion to its evidence.** A measurement from
110 legs is nearly all signal; one from 3 legs is one train's timetabling
habits. So the measured value is shrunk toward the European leg-weighted mean
of **@@SUPPLEMENT_PRIOR@@ %**:

```
supplement = measured × n/(n+@@SHRINKAGE@@) + European_mean × @@SHRINKAGE@@/(n+@@SHRINKAGE@@)
```

A hard sample-size cut-off would snap a country from its own value to the mean
on one extra leg; this degrades smoothly instead.

| Country | legs | measured % | weight | **supplement** | **quota** |
|---|---|---|---|---|---|
@@SPEED_ROWS@@

Worked example, **Austria**: 31.7 % measured over 56 legs, weight
56/(56+@@SHRINKAGE@@) = 0.85, so 0.85 × 31.7 + 0.15 × @@SUPPLEMENT_PRIOR@@ =
34.6 %. A leg the router puts at 60 minutes is scheduled at 81. Austria sits
well below the European figure because 56 legs is enough evidence to trust it —
the router is already slow through the Alps. **Norway** measured a similar
32.3 %, but over 3 legs: weight 0.23, so it lands near the mean at 46.4 %.

**No schema change is needed.** `track_buffer_quota_per` is already exactly
this multiplier and the router already applies it correctly — the legacy
placeholders it holds today (0.30–0.50) were doing this job uncalibrated,
which is why trips are currently only ~8 % fast rather than wildly wrong.
Seeding these values replaces a placeholder with a measurement in the same
column.

**Utilisation and punctuality are retained** per country with their sources,
and the theoretical formula above is still computed as a reference figure.
They are not multiplied into anything now — but they are precisely what a
"night trains receive improved priority" scenario would move, and that
scenario needs them calibrated and present.

**The weakest values are the ten countries with no ONTD legs** — EE, ES, FI,
GR, IE, LT, LU, LV, PT, UK — which take the European mean. Great Britain and
Spain are the two most likely to be understated: both run fast conventional
networks a loco-hauled sleeper cannot exploit, which is the French and Swedish
profile near 70 %, not the European 50 %. The fix is ONTD coverage, not more
reasoning.

### How the check works### How the check works

`ontd.route_legs` pairs, for every leg of every active ONTD route, the
**scheduled running time** from the real timetable with the **router's own
passage time** for the same leg:

```
implied_buffer_min = scheduled_running_min − routed_driving_min − routed_dynamics_min
```

`scheduled_running_min` is departure(from) → arrival(to), so it excludes dwell
and is directly comparable. The residual is attributed to countries by each
leg's `country_time_shares` and aggregated as
`Σ residual × share / Σ routed × share`.

**What this measures, and what it does not.** The residual is the gap between
our router and reality. Real buffer is only part of it. Also inside it: router
speed error (constant cruise speed per line class, so a systematic error
inflates every country together), operational time that is not buffer (border
control, locomotive changes, reversals, paths deliberately slowed so a train
arrives at a civilised hour rather than at 04:00), and legs where the real
train runs a different physical path than the router chose. So a country's
implied figure is an **upper bound** on its buffer.

Legs are excluded rather than cleaned: a residual above
@@MAX_RESIDUAL_PCT@@ %, or a leg shorter than @@MIN_LEG_MIN@@ minutes, is
treated as an operational stop, a path mismatch or dispatch noise,
and a *negative* residual — the real train faster than the router thinks
possible — is a router or path problem, never a negative buffer. Every
exclusion carries a named reason in `sources/ontd_buffer_legs.csv`, and the
excluded count is the honest measure of how dirty the sample is.

---

## 4. Dwell floor — @@MIN_DWELL@@ minutes per commercial stop

```
dwell(stop) = max(@@MIN_DWELL@@ min, operational_requirement(stop))
```

Uniform across all 28 countries and all stop sizes. For a sleeper service this
is a realistic floor: intermediate-station passenger exchange is small, doors
are attended, and the binding constraint is door release and dispatch rather
than passenger flow.

Two rules govern its use:

- **Dwell is additive, never multiplied by the buffer.** It is not running
  time, so a stop does not get longer because the country has a high buffer
  quota — which is why dwell sits outside the `(1 + buffer_pct)` term.
- **The floor is a floor, not a value.** Where a stop exists for another
  reason — locomotive or traction change at a border or electrification
  boundary, crew change, reversal, border control, watering, a scheduled
  crossing wait — the longer operational requirement **replaces** it rather
  than adding to it.

The effect on a night schedule is modest but not nothing: ten intermediate
stops carry 20 minutes of minimum dwell, roughly 2 % of a 15-hour journey, and
it lengthens the schedule without improving robustness — which is the argument
for keeping intermediate stops few on long-distance night services.

---

## 5. High-speed access — false everywhere

No country's high-speed lines are assumed open to a loco-hauled night train in
the base scenario. The flag is ANDed with the composition's own
`hsr_allowed`, so a high-speed-capable composition still needs the country to
permit it.

**Both this and the dwell floor stay per-country columns even though the value
is uniform**, and that is deliberate rather than leftover: a scenario
overrides per country, so the column is what lets one ask *what if Spanish
high-speed access opened to a night train* or *what if German dwell is really
four minutes* without a schema change. The uniform value is the calibration;
the column is the lever.

---

## 6. Confidence, ranked by how much it could move a result

1. **The delay coefficient (@@DELAY_COEF@@) is the least anchored number
   here.** It sets how hard poor punctuality pushes the buffer up, and it was
   chosen so the output lands in the 6–10 % band published practice occupies.
   The ONTD check is what settles it empirically — this is the single
   coefficient the night-train data should fix first.
2. **Punctuality thresholds are not harmonised across member states.** RMMS
   collects national figures and what counts as "on time" differs (5 minutes
   in some states, 15 in others), so a country reporting against a tight
   threshold looks worse than one reporting against a loose one, in an unknown
   direction. The strongest argument for replacing the term with a measured
   value rather than refining it analytically.
3. **Terrain is judgement, not measurement**, and the within-country spread
   often exceeds the between-country spread (§1). It is fit for country-level
   screening and for scaling one term of a placeholder consumption model; it is
   not fit for quoting a specific route's energy consumption.
4. **Utilisation is network-average, not corridor-specific.** A night train on
   a quiet secondary route in a dense country is over-padded by this model, and
   one on a saturated mainline in a sparse country is under-padded.
5. **Romania is a structural outlier, not just a high number.** With 19.7 % of
   long-distance services on time, no supplement in the 6–12 % range makes a
   published arrival credible. Treat the RO quota as a floor and flag any route
   depending on Romanian punctuality as carrying schedule risk this model does
   not capture.
6. **Night-specific effects are unmodelled and cut both ways.** Night paths
   face far less passenger-traffic conflict, arguing for a lower buffer than a
   day-derived figure; but they run through the maintenance window, when
   possessions, single-line working and diversions concentrate, arguing for
   more. The RMMS series is all-day and cannot separate them. The ONTD
   observation set can, and if night buffers come out systematically below the
   day-derived figures then the base term drops for every country.

---

## Sources

| source_id | Document | Publisher | Published | Data year | Link |
|---|---|---|---|---|---|
@@SOURCE_ROWS@@
"""

In [ ]:
CALIBRATION_TEMPLATE = r"""# Route Context — Calibration

Terrain, timetable buffer, dwell floor and high-speed access, calibrated per
country for @@N_COUNTRIES@@ European countries.

Generated by `02_route_context_calibration.ipynb` on @@GENDATE@@ — do not edit
by hand; re-run the notebooks (01 then 02, top to bottom) to regenerate this
document and the CSVs under `data/` and `seed/`. Calibration last reviewed end
to end @@REVIEWED@@.

**Provenance at a glance:** @@N_BENCHMARK@@ values from a pan-European
statistic, @@N_DERIVED@@ derived by documented arithmetic, @@N_ASSUMED@@
assumed with a band, across @@N_SOURCES@@ registered sources — of which
@@N_CITED@@ are cited by a per-country value and the rest underpin the method.
Nothing here is `sourced` in the sense the other three domains use it — there is no tariff
document for terrain or for a timetable supplement, which is why the ONTD
check in §3 matters more here than a cross-check does anywhere else.

---

@@BODY@@
"""

In [ ]:
# --- render ----------------------------------------------------------------
TOKENS = {
    "GENDATE": date.today().isoformat(),
    "REVIEWED": CALIBRATION_REVIEWED,
    "N_COUNTRIES": str(len(TERRAIN)),
    "N_BENCHMARK": str(_status_counts[BENCHMARK]),
    "N_DERIVED": str(_status_counts[DERIVED]),
    "N_ASSUMED": str(_status_counts[ASSUMED]),
    "N_SOURCES": str(len(register)),
    "N_CITED": str(len(cited)),
    "REF_TRAIN_T": f"{REF_TRAIN_T:g}",
    "SCORE_PER_M": f"{SCORE_PER_M_PER_KM:g}",
    "BAND_ROWS": BAND_ROWS,
    "TERRAIN_ROWS": TERRAIN_ROWS,
    "BASE_PCT": f"{BASE_PCT:g}",
    "UTIL_COEF": f"{UTIL_COEF:g}",
    "DELAY_COEF": f"{DELAY_COEF:g}",
    "UTIL_EU27": f"{UTIL_EU27:g}",
    "BUFFER_ROWS": BUFFER_ROWS,
    "BUFFER_MEDIAN": f"{statistics.median(list(buffer_pct.values())):.1f}",
    "BUFFER_MIN": f"{min(buffer_pct.values()):.1f}",
    "BUFFER_MAX": f"{max(buffer_pct.values()):.1f}",
    "MAX_RESIDUAL_PCT": "300",
    "MIN_DWELL": f"{MIN_DWELL_MIN:g}",
    "MIN_LEG_MIN": "10",
    "SPEED_ROWS": SPEED_ROWS,
    "SHRINKAGE": str(SUPPLEMENT_SHRINKAGE_LEGS),
    "SUPPLEMENT_PRIOR": (
        "—" if SUPPLEMENT_PRIOR is None else f"{SUPPLEMENT_PRIOR:.1f}"
    ),
    "ONTD_SECTION": ONTD_SECTION,
    "ONTD_N": str(ONTD_CORRELATIONS.get("n_countries", 0)),
    "SOURCE_ROWS": SOURCE_ROWS,
    "BODY": BODY.strip(),
}


def _render(text: str) -> str:
    for _ in range(3):
        for key, value in TOKENS.items():
            text = text.replace(f"@@{key}@@", value)
        if "@@" not in text:
            break
    return text


document = _render(CALIBRATION_TEMPLATE)
_left = sorted({t.split("@@")[0] for t in document.split("@@")[1::2]})
assert not _left, f"unsubstituted tokens: {_left}"

DOC_PATH.write_text(document, encoding="utf-8")
print(
    f"  {DOC_PATH.name}: {len(document.splitlines())} lines, {len(document):,} characters"
)

## Display

In [ ]:
# Display only — pandas is fine here, seed.py skips this cell.
import pandas as pd

pd.DataFrame(summary).set_index("country_code")